# 07 | Censored RUL and Calibrated Forecast Uncertainty

## Study objective

This notebook determines what can be stated honestly about remaining useful life when end of life is not observed, and whether future-SOH uncertainty remains calibrated for completely unseen SOFC cells.

The analysis has two linked decisions:

1. Can exact RUL be estimated from the available degradation trajectories?
2. Can the operational persistence forecast be accompanied by intervals that retain useful coverage across cells, horizons and redox regimes?

The notebook treats the physical cell as the independent engineering unit. It does not convert a stopped laboratory test into a false failure event.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sofc_health.targets.rul import add_rul_target

sns.set_theme(style="whitegrid", context="talk")


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing pyproject.toml. "
        "Start Jupyter Notebook from inside the project directory."
    )


ROOT = find_project_root(Path.cwd())
MODEL_DATA_PATH = ROOT / "data" / "processed" / "modeling_table.parquet"
NOTEBOOK_06_RESULTS = ROOT / "reports" / "tables" / "nested_ml" / "final_cross_cell"
FORECAST_PATH = NOTEBOOK_06_RESULTS / "final_selected_predictions.csv"
METRICS_PATH = NOTEBOOK_06_RESULTS / "final_selected_metrics.csv"
OUTPUT_DIRECTORY = ROOT / "reports" / "tables" / "uncertainty"
FIGURE_DIRECTORY = ROOT / "reports" / "figures" / "uncertainty"

for directory in (OUTPUT_DIRECTORY, FIGURE_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

required_files = [MODEL_DATA_PATH, FORECAST_PATH, METRICS_PATH]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    missing_text = "\n".join(f" - {path}" for path in missing_files)
    raise FileNotFoundError(
        "Notebook 07 requires the following files:\n"
        f"{missing_text}\n\n"
        "Run the data pipeline and optimized Notebook 06 first."
    )

table = (
    pd.read_parquet(MODEL_DATA_PATH)
    .sort_values(["cell_id", "assessment_index"])
    .reset_index(drop=True)
)
forecast_predictions = (
    pd.read_csv(FORECAST_PATH)
    .sort_values(["horizon", "cell_id", "assessment_index"])
    .reset_index(drop=True)
)
selected_metrics = pd.read_csv(METRICS_PATH)

TARGET = "soh_composite_pct"
EOL_THRESHOLD_PCT = 80.0
ALPHA = 0.10
NOMINAL_COVERAGE = 1 - ALPHA
HORIZONS = [1, 3, 5, 10]

required_prediction_columns = {
    "cell_id",
    "regime",
    "assessment_index",
    "future_assessment_index",
    "horizon",
    "actual_future_soh_pct",
    "persistence_prediction_pct",
}
missing_prediction_columns = required_prediction_columns.difference(forecast_predictions.columns)
if missing_prediction_columns:
    raise KeyError(
        f"Notebook 06 prediction columns are missing: {sorted(missing_prediction_columns)}"
    )
if forecast_predictions.duplicated(["cell_id", "horizon", "assessment_index"]).any():
    raise ValueError("Duplicate cell-horizon prediction rows were found.")

print("Python:", sys.version.split()[0])
print("Project root:", ROOT)
print("Modeling rows:", len(table))
print("Forecast rows:", len(forecast_predictions))
print("Cells:", sorted(table["cell_id"].unique()))
print("Horizons:", sorted(forecast_predictions["horizon"].unique()))
print("EOL threshold (%):", EOL_THRESHOLD_PCT)
print("Nominal interval coverage:", f"{NOMINAL_COVERAGE:.0%}")

## RUL as a censored first-passage problem

For an EOL threshold $\tau$, remaining useful life at assessment $k$ is the first future threshold-crossing time:

$$
RUL_{c,k}
=
\min\left\{h>0:SOH_{c,k+h}\le\tau\right\}
$$

If cell $c$ is observed only through assessment $L_c$ and does not reach EOL, its true failure assessment $T_c$ is unknown:

$$
T_c>L_c
$$

The exact RUL is therefore unavailable. The data provide only a lower bound:

$$
RUL_{c,k}=T_c-k>L_c-k
$$

`NaN` in the EOL column is meaningful censoring information. It must not be replaced with zero, the final assessment, a population mean or an extrapolated failure time.


In [ ]:
labeled = add_rul_target(
    table,
    threshold_pct=EOL_THRESHOLD_PCT,
)

eol_summary = (
    labeled.groupby("cell_id")
    .agg(
        first_assessment=("assessment_index", "min"),
        last_assessment=("assessment_index", "max"),
        observations=("assessment_index", "size"),
        minimum_observed_soh=(TARGET, "min"),
        final_observed_soh=(TARGET, "last"),
        eol_assessment=("eol_assessment", "first"),
        right_censored=("rul_right_censored", "first"),
    )
    .reset_index()
)

last_assessment_by_cell = labeled.groupby("cell_id")["assessment_index"].transform("max")

rul_lower_bounds = labeled[["cell_id", "assessment_index", TARGET, "rul_right_censored"]].copy()
rul_lower_bounds["last_observed_assessment"] = last_assessment_by_cell
rul_lower_bounds["observed_rul_lower_bound"] = (
    rul_lower_bounds["last_observed_assessment"] - rul_lower_bounds["assessment_index"]
)
rul_lower_bounds["bound_relation"] = np.where(
    rul_lower_bounds["rul_right_censored"],
    ">",
    "=",
)

eol_summary.to_csv(
    OUTPUT_DIRECTORY / "rul_censoring_summary.csv",
    index=False,
)
rul_lower_bounds.to_csv(
    OUTPUT_DIRECTORY / "rul_observed_lower_bounds.csv",
    index=False,
)

print("Observed EOL events:", eol_summary["eol_assessment"].notna().sum())
print("Right-censored cells:", eol_summary["right_censored"].sum())
display(eol_summary.round(3))

print("Example lower bounds at the midpoint assessment of each cell")
midpoint_examples = pd.concat(
    [frame.iloc[[len(frame) // 2]] for _, frame in rul_lower_bounds.groupby("cell_id", sort=True)],
    ignore_index=True,
)
display(
    midpoint_examples[
        [
            "cell_id",
            "assessment_index",
            "last_observed_assessment",
            "observed_rul_lower_bound",
            "bound_relation",
        ]
    ]
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

cell_order = ["N1", "N2", "N3", "N4", "N5", "N6", "R1", "R2"]
for position, cell_id in enumerate(cell_order):
    row = eol_summary.loc[eol_summary["cell_id"] == cell_id].iloc[0]
    color = "#0072B2" if cell_id.startswith("N") else "#D55E00"
    ax.hlines(
        y=position,
        xmin=row["first_assessment"],
        xmax=row["last_assessment"],
        color=color,
        linewidth=5,
        alpha=0.8,
    )
    ax.scatter(
        row["last_assessment"],
        position,
        marker=">",
        s=120,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        zorder=3,
    )

ax.set_yticks(range(len(cell_order)))
ax.set_yticklabels(cell_order)
ax.set_xlabel("Degradation assessment index")
ax.set_ylabel("Physical cell")
ax.set_title(
    "Observed lifetime windows at the 80% SOH threshold\n"
    "Arrowheads indicate right censoring, not observed failure",
    fontweight="bold",
)
ax.invert_yaxis()

timeline_path = FIGURE_DIRECTORY / "rul_censoring_timeline.png"
plt.tight_layout()
plt.savefig(timeline_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", timeline_path)

In [ ]:
threshold_records = []
event_records = []

for threshold in (75.0, 80.0, 85.0):
    threshold_labeled = add_rul_target(
        table,
        threshold_pct=threshold,
    )
    cell_status = (
        threshold_labeled.groupby("cell_id")
        .agg(
            eol_assessment=("eol_assessment", "first"),
            right_censored=("rul_right_censored", "first"),
        )
        .reset_index()
    )

    threshold_records.append(
        {
            "threshold_pct": threshold,
            "observed_eol_cells": int(cell_status["eol_assessment"].notna().sum()),
            "right_censored_cells": int(cell_status["right_censored"].sum()),
            "total_cells": len(cell_status),
        }
    )

    observed = cell_status.dropna(subset=["eol_assessment"]).copy()
    if not observed.empty:
        observed["threshold_pct"] = threshold
        event_records.append(observed)

threshold_sensitivity = pd.DataFrame(threshold_records)
observed_threshold_events = (
    pd.concat(event_records, ignore_index=True)
    if event_records
    else pd.DataFrame(
        columns=[
            "cell_id",
            "eol_assessment",
            "right_censored",
            "threshold_pct",
        ]
    )
)

threshold_sensitivity.to_csv(
    OUTPUT_DIRECTORY / "rul_threshold_sensitivity.csv",
    index=False,
)

print("EOL-threshold sensitivity")
display(threshold_sensitivity)
print("Observed threshold events, if any")
display(observed_threshold_events)

## Why an ordinary RUL model is not fitted

At the selected 80% threshold, all cells are censored. Consequently:

- An ordinary regression target for exact RUL does not exist.
- Treating the final assessment as EOL would bias lifetime downward.
- Dropping censored cells would remove the entire dataset.
- Kaplan-Meier survival would remain at one throughout the observed window, so median survival would be undefined.
- A parametric lifetime extrapolation would be driven by untestable distributional assumptions rather than observed failures.

The honest deliverable is therefore a censoring audit and observed lower bounds. Exact RUL prediction, RUL MAE and median lifetime are intentionally not reported.


## Cell-isolated conformal uncertainty

Notebook 06 retained persistence as the operational point-forecast benchmark. For origin assessment $k$ and horizon $h$:

$$
\widehat{SOH}_{c,k+h}=SOH_{c,k}
$$

For each test cell, calibration residuals are taken only from the other seven physical cells at the same forecast horizon:

$$
s_i=\left|SOH_i-\widehat{SOH}_i\right|
$$

The finite-sample corrected residual quantile $\hat q$ gives:

$$
\left[
\widehat{SOH}-\hat q,
\widehat{SOH}+\hat q
\right]
$$

Two interval constructions are reported:

1. **Pooled-cell interval:** uses all calibration residuals. This estimates marginal point-level coverage but treats temporally correlated rows as exchangeable.
2. **Cell-conservative sensitivity interval:** first calculates a high residual quantile within each calibration cell and then calibrates across those cell scores. With only seven calibration cells, this interval is intentionally conservative.

Neither method guarantees conditional coverage for every cell or regime. Coverage and interval width must therefore be reported together at cell, horizon and regime levels.


In [ ]:
def interval_score(actual, lower, upper, alpha):
    """Winkler interval score: narrow intervals are rewarded, misses are penalized."""
    actual = np.asarray(actual)
    lower = np.asarray(lower)
    upper = np.asarray(upper)
    width = upper - lower
    below_penalty = (2 / alpha) * (lower - actual) * (actual < lower)
    above_penalty = (2 / alpha) * (actual - upper) * (actual > upper)
    return width + below_penalty + above_penalty


def finite_sample_radius(actual, predicted, alpha):
    """Return the split-conformal absolute-residual radius."""
    scores = np.abs(np.asarray(actual) - np.asarray(predicted))
    if len(scores) == 0:
        raise ValueError("At least one calibration score is required.")

    quantile_level = min(
        1.0,
        np.ceil((len(scores) + 1) * (1 - alpha)) / len(scores),
    )
    return float(np.quantile(scores, quantile_level, method="higher"))


def build_cell_isolated_intervals(predictions, alpha=ALPHA):
    """Calibrate on other cells and evaluate on one completely isolated cell."""
    interval_frames = []
    audit_records = []

    for horizon in sorted(predictions["horizon"].unique()):
        horizon_data = predictions.loc[predictions["horizon"] == horizon]

        for test_cell in sorted(horizon_data["cell_id"].unique()):
            calibration = horizon_data.loc[horizon_data["cell_id"] != test_cell]
            evaluation = horizon_data.loc[horizon_data["cell_id"] == test_cell]

            calibration_cells = sorted(calibration["cell_id"].unique())
            if test_cell in calibration_cells:
                raise ValueError(f"Calibration leakage detected for {test_cell}.")

            pooled_radius = finite_sample_radius(
                calibration["actual_future_soh_pct"].to_numpy(),
                calibration["persistence_prediction_pct"].to_numpy(),
                alpha=alpha,
            )

            calibration_errors = calibration.assign(
                absolute_error=(
                    calibration["actual_future_soh_pct"] - calibration["persistence_prediction_pct"]
                ).abs()
            )
            within_cell_quantile = calibration_errors.groupby("cell_id")["absolute_error"].apply(
                lambda values: np.quantile(
                    values,
                    1 - alpha,
                    method="higher",
                )
            )
            conservative_radius = finite_sample_radius(
                within_cell_quantile.to_numpy(),
                np.zeros(len(within_cell_quantile)),
                alpha=alpha,
            )

            audit_records.append(
                {
                    "horizon": horizon,
                    "test_cell": test_cell,
                    "calibration_cells": len(calibration_cells),
                    "calibration_rows": len(calibration),
                    "evaluation_rows": len(evaluation),
                    "test_cell_in_calibration": False,
                    "pooled_radius": pooled_radius,
                    "cell_conservative_radius": conservative_radius,
                }
            )

            for method, radius in {
                "pooled_cell": pooled_radius,
                "cell_conservative": conservative_radius,
            }.items():
                center = evaluation["persistence_prediction_pct"].to_numpy()
                lower = center - radius
                upper = center + radius
                frame = evaluation[
                    [
                        "cell_id",
                        "regime",
                        "assessment_index",
                        "future_assessment_index",
                        "horizon",
                        "actual_future_soh_pct",
                        "persistence_prediction_pct",
                    ]
                ].copy()
                frame["interval_method"] = method
                frame["alpha"] = alpha
                frame["nominal_coverage"] = 1 - alpha
                frame["radius_soh_pct"] = radius
                frame["lower_soh_pct"] = lower
                frame["upper_soh_pct"] = upper
                frame["covered"] = frame["actual_future_soh_pct"].ge(
                    frame["lower_soh_pct"]
                ) & frame["actual_future_soh_pct"].le(frame["upper_soh_pct"])
                frame["interval_width_soh_pct"] = upper - lower
                frame["interval_score"] = interval_score(
                    frame["actual_future_soh_pct"],
                    lower,
                    upper,
                    alpha,
                )
                frame["miss_distance_soh_pct"] = np.maximum.reduce(
                    [
                        lower - frame["actual_future_soh_pct"].to_numpy(),
                        frame["actual_future_soh_pct"].to_numpy() - upper,
                        np.zeros(len(frame)),
                    ]
                )
                interval_frames.append(frame)

    return (
        pd.concat(interval_frames, ignore_index=True),
        pd.DataFrame(audit_records),
    )


conformal_predictions, calibration_audit = build_cell_isolated_intervals(
    forecast_predictions,
    alpha=ALPHA,
)

expected_interval_rows = 2 * len(forecast_predictions)
if len(conformal_predictions) != expected_interval_rows:
    raise ValueError(
        f"Expected {expected_interval_rows} interval rows, received {len(conformal_predictions)}."
    )
if calibration_audit["test_cell_in_calibration"].any():
    raise ValueError("A held-out test cell entered interval calibration.")
if not calibration_audit["calibration_cells"].eq(7).all():
    raise ValueError("Each fold must use exactly seven calibration cells.")

conformal_predictions.to_csv(
    OUTPUT_DIRECTORY / "cell_isolated_conformal_predictions.csv",
    index=False,
)
calibration_audit.to_csv(
    OUTPUT_DIRECTORY / "conformal_calibration_audit.csv",
    index=False,
)

print("Cell-isolated conformal calibration completed.")
print("Interval rows:", len(conformal_predictions))
print("Calibration folds:", len(calibration_audit))
print("Cell-leakage violations:", calibration_audit["test_cell_in_calibration"].sum())
display(calibration_audit.head(8).round(3))

In [ ]:
def summarize_intervals(group):
    return pd.Series(
        {
            "observations": len(group),
            "coverage": group["covered"].mean(),
            "mean_interval_width": group["interval_width_soh_pct"].mean(),
            "median_interval_width": group["interval_width_soh_pct"].median(),
            "mean_interval_score": group["interval_score"].mean(),
            "misses": int((~group["covered"]).sum()),
            "maximum_miss_distance": group["miss_distance_soh_pct"].max(),
        }
    )


def summarize_by(frame, group_columns):
    """Summarize groups without relying on version-specific groupby.apply behavior."""
    records = []
    group_key = group_columns[0] if len(group_columns) == 1 else group_columns

    for key, group in frame.groupby(group_key, sort=True, observed=True):
        key_values = (key,) if len(group_columns) == 1 else key
        record = dict(zip(group_columns, key_values, strict=True))
        record.update(summarize_intervals(group).to_dict())
        records.append(record)

    return pd.DataFrame(records)


overall_interval_summary = summarize_by(
    conformal_predictions,
    ["interval_method"],
)
horizon_interval_summary = summarize_by(
    conformal_predictions,
    ["interval_method", "horizon"],
)
regime_interval_summary = summarize_by(
    conformal_predictions,
    ["interval_method", "horizon", "regime"],
)
cell_interval_summary = summarize_by(
    conformal_predictions,
    ["interval_method", "horizon", "cell_id", "regime"],
)

overall_interval_summary.to_csv(
    OUTPUT_DIRECTORY / "conformal_overall_summary.csv",
    index=False,
)
horizon_interval_summary.to_csv(
    OUTPUT_DIRECTORY / "conformal_horizon_summary.csv",
    index=False,
)
regime_interval_summary.to_csv(
    OUTPUT_DIRECTORY / "conformal_regime_summary.csv",
    index=False,
)
cell_interval_summary.to_csv(
    OUTPUT_DIRECTORY / "conformal_cell_summary.csv",
    index=False,
)

print("Overall coverage and sharpness")
display(overall_interval_summary.round(3))
print("Coverage and sharpness by horizon")
display(horizon_interval_summary.round(3))
print("Coverage and sharpness by horizon and regime")
display(regime_interval_summary.round(3))

In [ ]:
coverage_curve_records = []

for nominal_coverage in (0.50, 0.70, 0.80, 0.90, 0.95):
    curve_predictions, _ = build_cell_isolated_intervals(
        forecast_predictions,
        alpha=1 - nominal_coverage,
    )
    pooled_curve = curve_predictions.loc[curve_predictions["interval_method"] == "pooled_cell"]

    for horizon, frame in pooled_curve.groupby("horizon"):
        coverage_curve_records.append(
            {
                "nominal_coverage": nominal_coverage,
                "horizon": horizon,
                "empirical_coverage": frame["covered"].mean(),
                "mean_interval_width": frame["interval_width_soh_pct"].mean(),
            }
        )

coverage_curve = pd.DataFrame(coverage_curve_records)
coverage_curve.to_csv(
    OUTPUT_DIRECTORY / "conformal_coverage_curve.csv",
    index=False,
)

print("Pooled-cell empirical calibration curve")
display(coverage_curve.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for horizon in HORIZONS:
    horizon_curve = coverage_curve.loc[coverage_curve["horizon"] == horizon]
    axes[0].plot(
        horizon_curve["nominal_coverage"],
        horizon_curve["empirical_coverage"],
        marker="o",
        linewidth=2,
        label=f"h = {horizon}",
    )

axes[0].plot(
    [0.45, 1.0],
    [0.45, 1.0],
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Ideal calibration",
)
axes[0].set_xlim(0.48, 0.97)
axes[0].set_ylim(0.48, 1.0)
axes[0].set_xlabel("Nominal coverage")
axes[0].set_ylabel("Empirical coverage")
axes[0].set_title("Coverage calibration", fontweight="bold")
axes[0].legend(fontsize=10)

for method, frame in horizon_interval_summary.groupby("interval_method"):
    label = method.replace("_", " ").title()
    axes[1].plot(
        frame["horizon"],
        frame["mean_interval_width"],
        marker="o",
        linewidth=2.2,
        label=label,
    )

axes[1].set_xticks(HORIZONS)
axes[1].set_xlabel("Forecast horizon, future assessments")
axes[1].set_ylabel("Mean interval width, SOH percentage points")
axes[1].set_title("Uncertainty growth with horizon", fontweight="bold")
axes[1].legend(fontsize=10)

fig.suptitle(
    "Calibration and sharpness of cell-isolated SOH intervals",
    fontsize=17,
    fontweight="bold",
)
plt.tight_layout()

calibration_figure_path = FIGURE_DIRECTORY / "conformal_calibration_and_width.png"
plt.savefig(calibration_figure_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", calibration_figure_path)

In [ ]:
diagnostic_cases = [
    {
        "cell_id": "N2",
        "horizon": 10,
        "title": "Lowest-coverage regular cell",
    },
    {
        "cell_id": "R1",
        "horizon": 10,
        "title": "Lowest-coverage randomized-redox cell",
    },
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)

for ax, case in zip(axes, diagnostic_cases, strict=True):
    frame = conformal_predictions.loc[
        (conformal_predictions["interval_method"] == "pooled_cell")
        & (conformal_predictions["cell_id"] == case["cell_id"])
        & (conformal_predictions["horizon"] == case["horizon"])
    ].sort_values("future_assessment_index")

    if frame.empty:
        raise ValueError(f"Missing interval case: {case}")

    x = frame["future_assessment_index"].to_numpy()
    actual = frame["actual_future_soh_pct"].to_numpy()
    point = frame["persistence_prediction_pct"].to_numpy()
    lower = frame["lower_soh_pct"].to_numpy()
    upper = frame["upper_soh_pct"].to_numpy()

    ax.fill_between(
        x,
        lower,
        upper,
        color="#56B4E9",
        alpha=0.28,
        label="90% pooled-cell interval",
    )
    ax.plot(
        x,
        actual,
        color="black",
        linewidth=2.3,
        marker="o",
        markersize=3.5,
        label="Observed future SOH",
    )
    ax.plot(
        x,
        point,
        color="#0072B2",
        linewidth=2,
        linestyle="--",
        label="Persistence forecast",
    )

    coverage = frame["covered"].mean()
    width = frame["interval_width_soh_pct"].mean()
    ax.set_title(
        f"{case['title']}\nCell {case['cell_id']}, h = {case['horizon']}",
        fontweight="bold",
    )
    ax.set_xlabel("Future assessment index")
    ax.set_ylabel("Composite SOH (%)")
    ax.text(
        0.03,
        0.04,
        f"Coverage: {coverage:.1%}\nMean width: {width:.2f} SOH points",
        transform=ax.transAxes,
        fontsize=10,
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": "white",
            "edgecolor": "lightgray",
            "alpha": 0.9,
        },
    )

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=3,
    frameon=False,
    fontsize=10,
)
fig.suptitle(
    "Cell-isolated persistence intervals at the longest horizon",
    fontsize=17,
    fontweight="bold",
    y=1.10,
)
plt.tight_layout()

trajectory_figure_path = FIGURE_DIRECTORY / "conformal_interval_case_studies.png"
plt.savefig(trajectory_figure_path, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", trajectory_figure_path)

In [ ]:
pooled_cell_summary = cell_interval_summary.loc[
    cell_interval_summary["interval_method"] == "pooled_cell"
].copy()

reliability_audit = pooled_cell_summary.merge(
    selected_metrics[
        [
            "cell_id",
            "horizon",
            "selected_model",
            "selected_representation",
            "mae_skill",
        ]
    ],
    on=["cell_id", "horizon"],
    how="left",
    validate="one_to_one",
)
reliability_audit["coverage_gap"] = reliability_audit["coverage"] - NOMINAL_COVERAGE
reliability_audit["material_undercoverage"] = reliability_audit["coverage_gap"] < -0.05
reliability_audit.to_csv(
    OUTPUT_DIRECTORY / "uncertainty_reliability_audit.csv",
    index=False,
)

print("Lowest-coverage cell and horizon combinations")
display(
    reliability_audit.sort_values(
        ["coverage", "maximum_miss_distance"],
        ascending=[True, False],
    )[
        [
            "cell_id",
            "regime",
            "horizon",
            "coverage",
            "mean_interval_width",
            "mean_interval_score",
            "maximum_miss_distance",
            "selected_model",
            "selected_representation",
            "mae_skill",
            "material_undercoverage",
        ]
    ]
    .head(12)
    .round(3)
)

## Does reliability change across the observed degradation trajectory?

Aggregate coverage can hide a time-series failure mode. An interval may work early in a test, when the cell resembles its initial state, but become unreliable later as degradation accumulates and the predictors move away from the calibration distribution.

Each forecast origin is therefore assigned to the early, middle or late third of that cell's available trajectory. This is a relative stage index, not operating time. The diagnostic asks whether coverage deteriorates as the observed test progresses.


In [ ]:
lifecycle_intervals = conformal_predictions.copy()
lifecycle_group = lifecycle_intervals.groupby(
    ["interval_method", "cell_id", "horizon"],
    observed=True,
)["assessment_index"]
lifecycle_start = lifecycle_group.transform("min")
lifecycle_end = lifecycle_group.transform("max")
lifecycle_span = (lifecycle_end - lifecycle_start).replace(0, np.nan)

lifecycle_intervals["observed_lifecycle_fraction"] = (
    (lifecycle_intervals["assessment_index"] - lifecycle_start) / lifecycle_span
).fillna(0.0)
lifecycle_intervals["observed_lifecycle_stage"] = pd.cut(
    lifecycle_intervals["observed_lifecycle_fraction"],
    bins=[-0.001, 1 / 3, 2 / 3, 1.001],
    labels=["early", "middle", "late"],
    include_lowest=True,
)

lifecycle_stage_summary = summarize_by(
    lifecycle_intervals,
    [
        "interval_method",
        "horizon",
        "observed_lifecycle_stage",
    ],
)
lifecycle_stage_summary.to_csv(
    OUTPUT_DIRECTORY / "conformal_lifecycle_stage_summary.csv",
    index=False,
)

pooled_lifecycle = lifecycle_stage_summary.loc[
    lifecycle_stage_summary["interval_method"] == "pooled_cell"
].copy()
stage_order = ["early", "middle", "late"]
coverage_matrix = pooled_lifecycle.pivot(
    index="horizon",
    columns="observed_lifecycle_stage",
    values="coverage",
).reindex(index=HORIZONS, columns=stage_order)
width_matrix = pooled_lifecycle.pivot(
    index="horizon",
    columns="observed_lifecycle_stage",
    values="mean_interval_width",
).reindex(index=HORIZONS, columns=stage_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(
    coverage_matrix,
    annot=True,
    fmt=".2f",
    vmin=0.60,
    vmax=1.00,
    cmap="RdYlGn",
    cbar_kws={"label": "Empirical coverage"},
    ax=axes[0],
)
axes[0].set_title("Coverage", fontweight="bold")
axes[0].set_xlabel("Observed lifecycle stage")
axes[0].set_ylabel("Forecast horizon")

sns.heatmap(
    width_matrix,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={"label": "Mean width, SOH points"},
    ax=axes[1],
)
axes[1].set_title("Interval width", fontweight="bold")
axes[1].set_xlabel("Observed lifecycle stage")
axes[1].set_ylabel("Forecast horizon")

fig.suptitle(
    "Pooled-cell uncertainty across the observed degradation trajectory",
    fontsize=17,
    fontweight="bold",
)
plt.tight_layout()

lifecycle_figure_path = FIGURE_DIRECTORY / "conformal_lifecycle_stage_heatmaps.png"
plt.savefig(lifecycle_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Lifecycle-stage reliability")
display(lifecycle_stage_summary.round(3))
print("Figure saved to:", lifecycle_figure_path)

## Engineering decision framework

The notebook supports three different claims, and they must not be confused:

1. **Observed survival:** every right-censored cell survived beyond its final recorded assessment.
2. **Future-SOH uncertainty:** persistence forecasts can be accompanied by empirically calibrated intervals, subject to the reported cell and regime limitations.
3. **Exact RUL:** not validated because the selected EOL event was never observed.

A deployable uncertainty layer must satisfy more than aggregate coverage. It should maintain acceptable coverage at each relevant horizon, avoid systematic undercoverage for randomized-redox cells, and remain reasonably sharp. If coverage deteriorates or intervals become operationally too wide, the system should report insufficient evidence rather than a precise health or lifetime estimate.

The numerical conclusion should be finalized only after reviewing:

- Overall coverage and interval width
- Horizon-level coverage
- Regime-level coverage
- Worst-cell coverage and miss distance
- The difference between pooled and cell-conservative intervals

Notebook 08 will use these results for ablation, robustness checks and the final engineering claim matrix.


In [ ]:
pooled_overall = overall_interval_summary.loc[
    overall_interval_summary["interval_method"] == "pooled_cell"
].iloc[0]
conservative_overall = overall_interval_summary.loc[
    overall_interval_summary["interval_method"] == "cell_conservative"
].iloc[0]
pooled_horizons = horizon_interval_summary.loc[
    horizon_interval_summary["interval_method"] == "pooled_cell"
]
pooled_cells = cell_interval_summary.loc[cell_interval_summary["interval_method"] == "pooled_cell"]

minimum_horizon_coverage = pooled_horizons["coverage"].min()
minimum_cell_horizon_coverage = pooled_cells["coverage"].min()
diagnostic_coverage_supported = (
    pooled_overall["coverage"] >= NOMINAL_COVERAGE
    and minimum_horizon_coverage >= NOMINAL_COVERAGE - 0.05
)

decision_register = pd.DataFrame(
    [
        {
            "decision": "Exact RUL estimation",
            "evidence": f"{int(eol_summary['eol_assessment'].notna().sum())} observed EOL events at {EOL_THRESHOLD_PCT:.0f}% SOH",
            "screening_rule": "At least one observed failure is required before empirical lifetime validation",
            "status": "NOT ESTIMABLE",
        },
        {
            "decision": "Pooled-cell interval calibration",
            "evidence": (
                f"overall coverage={pooled_overall['coverage']:.3f}; "
                f"minimum horizon coverage={minimum_horizon_coverage:.3f}"
            ),
            "screening_rule": (
                f"overall coverage >= {NOMINAL_COVERAGE:.2f} and no horizon more than 0.05 below nominal"
            ),
            "status": (
                "SUPPORTED FOR OFFLINE DIAGNOSTIC USE"
                if diagnostic_coverage_supported
                else "UNDERCOVERED"
            ),
        },
        {
            "decision": "Worst cell-horizon reliability",
            "evidence": f"minimum pooled coverage={minimum_cell_horizon_coverage:.3f}",
            "screening_rule": "Report explicitly because marginal coverage does not guarantee cell-level coverage",
            "status": "REVIEW REQUIRED",
        },
        {
            "decision": "Cell-conservative sensitivity",
            "evidence": (
                f"coverage={conservative_overall['coverage']:.3f}; "
                f"mean width={conservative_overall['mean_interval_width']:.3f} SOH points"
            ),
            "screening_rule": "Use as a robustness bound, not as proof of deployment readiness",
            "status": "SENSITIVITY RESULT",
        },
        {
            "decision": "Operational sharpness",
            "evidence": f"pooled mean width={pooled_overall['mean_interval_width']:.3f} SOH points",
            "screening_rule": "Requires a stakeholder-defined maximum useful width",
            "status": "REQUIREMENT NOT DEFINED",
        },
        {
            "decision": "Field deployment",
            "evidence": "Eight laboratory cells, assessment-index time, regime shift and no field telemetry",
            "screening_rule": "External field validation and operating-hour alignment are required",
            "status": "NOT VALIDATED",
        },
    ]
)
decision_register.to_csv(
    OUTPUT_DIRECTORY / "notebook07_decision_register.csv",
    index=False,
)

print("Notebook 07 decision register")
display(decision_register)
print()
print("Final technical conclusion")
print(
    "Exact RUL is not identifiable at the selected threshold because all "
    "cells are right-censored. The defensible output is an observed RUL "
    "lower bound plus cell-isolated uncertainty for future composite SOH."
)
print(
    "The uncertainty result is an offline laboratory diagnostic, not a "
    "deployment claim. Review horizon, regime, cell and lifecycle-stage "
    "coverage together with interval width before accepting it."
)

## Research handoff to Notebook 08: EIS mechanisms and health coupling

Notebook 07 identified a structured uncertainty failure rather than uniform random error. Regular-redox cells were comparatively stable, while the randomized-redox cells, especially R1, showed reversible early behaviour that persistence and fixed-width intervals could not represent. This motivates a physics-based investigation of whether impedance measurements contain information about those changes.

Notebook 08 will test four hypotheses:

1. Increasing polarization resistance is associated with declining IV and transient performance within a physical cell.
2. Ohmic and polarization resistance carry different degradation information and should not be treated as interchangeable health indicators.
3. Changes in ECM and DRT features help distinguish gradual degradation from reversible redox-related excursions.
4. EIS features add future-health information beyond current SOH, assessment index and persistence.

To separate degradation effects from manufacturing differences, EIS variables will be expressed relative to each cell's initial diagnostic:

$$
\Delta x_{c,k}^{rel}
=
100\left(
\frac{x_{c,k}}{x_{c,1}}-1
\right)
$$

Cross-modal relationships will then be evaluated with a cell-aware model:

$$
\Delta SOH_{c,k}
=
\beta_0
+\beta_1\Delta R_{ohm,c,k}
+\beta_2\Delta R_{pol,c,k}
+\beta_3G_c
+\beta_4\Delta R_{pol,c,k}G_c
+u_c
+\varepsilon_{c,k}
$$

Here, $G_c$ represents redox regime and $u_c$ represents persistent cell-specific variation. The interaction term tests whether polarization changes have a different association with health under randomized redox operation.

The horizontal axis will remain the degradation assessment index. It will not be described as operating hours or true lifetime because reliable operating-time intervals are unavailable.

Notebook 08 will use `eis.parquet`, `modeling_table.parquet` and the uncertainty diagnostics produced here. Associations will be reported within cells and by regime. Causal mechanism claims will require physical consistency, temporal ordering, robustness across cells and improvement over simpler explanations.


## Reproducibility outputs

The notebook creates the following tables under `reports/tables/uncertainty/`:

- `rul_censoring_summary.csv`
- `rul_observed_lower_bounds.csv`
- `rul_threshold_sensitivity.csv`
- `cell_isolated_conformal_predictions.csv`
- `conformal_calibration_audit.csv`
- `conformal_overall_summary.csv`
- `conformal_horizon_summary.csv`
- `conformal_regime_summary.csv`
- `conformal_cell_summary.csv`
- `conformal_coverage_curve.csv`
- `uncertainty_reliability_audit.csv`
- `conformal_lifecycle_stage_summary.csv`
- `notebook07_decision_register.csv`

Figures are saved under `reports/figures/uncertainty/`.

Run the notebook from top to bottom after Notebook 06 has produced its final prediction and metric tables.
